The goal of this notebook is to model the topics of our dataset with BERTopic

In [ ]:
from bertopic import BERTopic
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import polars as pl

In [ ]:
data = pl.read_parquet("works_pre_topics.parquet")

In [ ]:
data = data.with_columns(
    title_abstract = pl.col("title") + pl.lit(". ") + pl.col("abstract")
)

In [ ]:
data = data.filter(pl.col("title_abstract").is_not_null())

In [ ]:
data = data.filter(pl.col("title_abstract") != "")

In [ ]:
title_abstract = data["title_abstract"].to_list()
dates = data["year"]

# Training

In [ ]:
from bertopic import BERTopic
from sentence_transformers import SentenceTransformer
from umap import UMAP
from sklearn.feature_extraction.text import CountVectorizer # for deleting the stop words
from sklearn.feature_extraction.text import ENGLISH_STOP_WORDS
from nltk.tokenize import word_tokenize

umap_model = UMAP(n_neighbors=15, n_components=5, metric='cosine', random_state=42)
embedding_model = SentenceTransformer("all-MiniLM-L6-v2", device="cuda")

In [ ]:
custom_stop_words = ["current",
                     "board",
                     "editorial",
                     "thank",
                     "contents",
                     "table",
                     "availability",
                     "matter",
                     "talking",
                     "question",
                     "olde r",
                     "erratum",
                     "eradication",
                     "approval",
                     "global",
                     "correction",
                     "written",
                     "issue",
                     "information",
                     "publication",
                     "publishing",
                     "implementing",
                     "noticeboard",
                     "issue",
                     "problems"]
all_stop_words = list(ENGLISH_STOP_WORDS.union(custom_stop_words))
vectorizer_model = CountVectorizer(
    stop_words=all_stop_words,
    lowercase=True,
    token_pattern=r"(?u)\b\w\w+\b"
)



def clean_text(text):
    tokens = word_tokenize(text.lower())
    return ' '.join([word for word in tokens if word not in all_stop_words])

cleaned_title_abstract = [clean_text(doc) for doc in title_abstract]

In [ ]:
topic_modelling = BERTopic(embedding_model=embedding_model, umap_model=umap_model, language="english", vectorizer_model=vectorizer_model)
topics, probs = topic_modelling.fit_transform(cleaned_title_abstract)

In [ ]:
topic_modelling.save("bertopic_model_pharmacology")

In [ ]:
data = data.with_columns(
    pl.Series("topic", topics)
)


In [ ]:
data.write_parquet("works_post_topics.parquet")

# Result analysis

In [ ]:
topic_modelling = BERTopic.load("bertopic_model_pharmacology", embedding_model=embedding_model)

In [ ]:
test = topic_modelling.get_topics()
test = pd.DataFrame(test)
test.to_csv("topic_words.csv")

In [ ]:
topic_modelling.visualize_topics()

In [ ]:
topic_modelling.visualize_barchart(top_n_topics=400)


In [ ]:
data_filtered = data.filter(pl.col("topic").is_in([1, 3, 4, 13, 97]))
timestamps = pd.to_datetime(dates, format="%Y").to_list()
topics_over_time = topic_modelling.topics_over_time(
    docs = data_filtered["abstract"].to_list(),
    topics = data_filtered["topic"].to_list(),
    timestamps = pd.to_datetime(data_filtered["year"], format="%Y").to_list(),
    global_tuning=True,  # ou False selon la granularité voulue
    evolution_tuning=True,  # pour détecter des variations dans le vocabulaire
    nr_bins=20  # nombre de périodes de temps (buckets)
)
# convertir topics_over_time en df et filtrer un ou qq topics
topic_modelling.visualize_topics_over_time(topics_over_time)

In [ ]:
# pd.crosstab pour analyse catégorielle (puis heatmap de la crosstab)

In [ ]:
# aide à réduire le nombre de topics
topic_modelling.visualize_hierarchy(top_n_topics=200)

In [ ]:
topic_modelling.visualize_heatmap(n_clusters=20, width=1000, height=1000)